# **Faster R-CNN Implementation**

### **1. Imports**

In [7]:
import pandas as pd
import numpy as np
import os, math
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import json
from collections import defaultdict
from pathlib import Path
from PIL import Image
import torch
from torchvision.transforms.functional import to_tensor
from torch.utils.data import DataLoader
from tqdm import tqdm

In [8]:
print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Using device: NVIDIA GeForce RTX 3060


### **2. Load pretrained model**

In [9]:
def get_model(num_classes=5):
    model = fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1,
        min_size=640, max_size=640
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

### **3. Dataset Class**

In [10]:
class MioLocDataset(torch.utils.data.Dataset):
    def __init__(self, root, split, train=False):
        self.root, self.train = Path(root), train
        data = json.loads((self.root / f'coco_{split}.json').read_text())
        self.images = data['images']
        self.anns = defaultdict(list)
        for a in data['annotations']:
            self.anns[a['image_id']].append(a)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        info = self.images[i]
        img = to_tensor(Image.open(self.root / info['file_name']).convert('RGB'))
        anns = self.anns[info['id']]
        boxes = torch.tensor([[a['bbox'][0], a['bbox'][1],
                               a['bbox'][0] + a['bbox'][2], a['bbox'][1] + a['bbox'][3]]
                              for a in anns], dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor([a['category_id'] for a in anns], dtype=torch.int64)
        if self.train and torch.rand(1) < 0.5:  # horizontal flip augmentation
            img = img.flip(-1)
            boxes[:, [0, 2]] = img.shape[-1] - boxes[:, [2, 0]]
        return img, {'boxes': boxes, 'labels': labels}

def collate(batch):
    return tuple(zip(*batch))

### **4. Training Loop**

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = r"C:\Users\shif\Uni_Workspace\live-vehicle-classification\data\MIO-TCD-Localization\MIO-TCD-Localization\processed_mio_localization"
CKPT = 'frcnn_latest.pt'
EPOCHS, BATCH, SAVE_EVERY = 3 , 4, 500   # save every 500 iterations

train_ds = MioLocDataset(root, 'train', train=True)

model = get_model().to(device)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)
scaler = torch.amp.GradScaler('cuda')

def save_ckpt(epoch, it):
    tmp = CKPT + '.tmp'
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'epoch': epoch, 'iter': it}, tmp)
    os.replace(tmp, CKPT)   # atomic, so a crash mid-save can't corrupt the checkpoint

def make_batches(epoch):
    # same shuffle order for a given epoch, so resuming mid-epoch is exact
    g = torch.Generator().manual_seed(epoch)
    idx = torch.randperm(len(train_ds), generator=g).tolist()
    return [idx[i:i + BATCH] for i in range(0, len(idx), BATCH)]

start_epoch, start_iter = 0, 0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler'])
    start_epoch, start_iter = ck['epoch'], ck['iter']
    print(f'Resumed from epoch {start_epoch}, iteration {start_iter}')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    batches = make_batches(epoch)
    skip = start_iter if epoch == start_epoch else 0
    loader = DataLoader(train_ds, batch_sampler=batches[skip:],
                        num_workers=0, collate_fn=collate)

    bar = tqdm(loader, total=len(batches), initial=skip, desc=f'epoch {epoch+1}/{EPOCHS}')
    for step, (imgs, targets) in enumerate(bar, start=skip):
        imgs = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.autocast('cuda'):
            loss = sum(model(imgs, targets).values())
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        bar.set_postfix(loss=f'{loss.item():.3f}')

        if (step + 1) % SAVE_EVERY == 0:
            save_ckpt(epoch, step + 1)

    scheduler.step()
    save_ckpt(epoch + 1, 0)
    torch.save(model.state_dict(), f'frcnn_epoch{epoch}.pt')

epoch 3/3: 100%|██████████| 22000/22000 [2:26:06<00:00,  2.51it/s, loss=0.152]  


In [12]:
print(torch.cuda.is_available(), next(model.parameters()).device)

True cuda:0


In [13]:
model.eval()
with torch.no_grad():
    out = model([img.to(device)])[0]   # dict with boxes, labels, scores

NameError: name 'img' is not defined

### **5. Model Evaluation**

In [ ]:
import matplotlib.pyplot as plt
